In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
ls "/content/drive/MyDrive/Colab Notebooks/Cross-Crop Transferability Index"

compute_cti.py                           data/
crop_list.txt                            hybrid_model.py
cti_heatmap_20260222_231502.png          models/
cti_summary_20260222_231502.csv          __pycache__/
cti_transfer_matrix_20260222_231502.csv  Untitled.ipynb


In [ ]:
!ls "/content/drive/MyDrive/Colab Notebooks/Cross-Crop Transferability Index/data/test"
!ls "/content/drive/MyDrive/Colab Notebooks/Cross-Crop Transferability Index/models"

cauliflower  Citrus	  large_cardamom  mung_bean  rice    strawberry
chili	     kidney_bean  maize		  onion      sesame
ConvNeXt_V2_Atto_best.pth	       Mobilenet_v2_Citrus.pth
Efficienet_net_V2model_Strawberry.pth  MobilenetV2_Rice_model.pth
Hybrid_chilli.pth		       ResNet18_Weights_Strawberry.pth
HybridMaizemodel.pth		       squeezenet_Rajma_ND.pth
Hybridmungbean.pth		       Squeez_suffle_cardamom_Hybrid.pth
Hybrid_Onion_model.pth		       Sufflenet_v2_strwaberry.pth
hybrid_sesame.pth		       v2plantnet_state_dict.pth
Mobilenetv2_cauliflower.pth


In [ ]:
# Cell 1: Show real structure (including hidden files)
!ls -la "/content/drive/MyDrive/Colab Notebooks/Cross-Crop Transferability Index/data/test/cauliflower"
!ls -la "/content/drive/MyDrive/Colab Notebooks/Cross-Crop Transferability Index/data/test/cauliflower/Healthy"   # pick one class

total 16
drwx------ 2 root root 4096 Feb 22 21:23  Alternaria
drwx------ 2 root root 4096 Feb 22 21:23  Healthy
drwx------ 2 root root 4096 Feb 22 21:23 'Phosphorous Deficiency'
drwx------ 2 root root 4096 Feb 22 21:23  Phytoxicity
total 11973
-rw------- 1 root root  47920 Jun 10  2025 HCF109.jpg
-rw------- 1 root root  48447 Jun 10  2025 HCF11.jpg
-rw------- 1 root root  46071 Jun 10  2025 HCF130.jpg
-rw------- 1 root root  40749 Jun 10  2025 HCF150.jpg
-rw------- 1 root root  41338 Jun 10  2025 HCF158.jpg
-rw------- 1 root root  48473 Jun 10  2025 HCF15.jpg
-rw------- 1 root root  41277 Jun 10  2025 HCF163.jpg
-rw------- 1 root root  40678 Jun 10  2025 HCF165.jpg
-rw------- 1 root root  39496 Jun 10  2025 HCF169.jpg
-rw------- 1 root root  51729 Jun 10  2025 HCF174.jpg
-rw------- 1 root root  51565 Jun 10  2025 HCF175.jpg
-rw------- 1 root root  51567 Jun 10  2025 HCF187.jpg
-rw------- 1 root root  38260 Jun 10  2025 HCF188.jpg
-rw------- 1 root root  37355 Jun 10  2025 HCF196.jpg
-r

In [ ]:
# Cell 2: Count images per class (quick check)
import os
base = "/content/drive/MyDrive/Colab Notebooks/Cross-Crop Transferability Index/data/test"
for crop in ['cauliflower', 'chili', 'large_cardamom']:   # add others
    crop_path = os.path.join(base, crop)
    if not os.path.exists(crop_path):
        print(f"{crop} folder missing!")
        continue
    print(f"\n{crop}:")
    for cls in sorted(os.listdir(crop_path)):
        cls_path = os.path.join(crop_path, cls)
        if os.path.isdir(cls_path):
            imgs = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png','.bmp','.tiff'))]
            print(f"  {cls:25} → {len(imgs)} images")


cauliflower:
  Alternaria                → 301 images
  Healthy                   → 141 images
  Phosphorous Deficiency    → 204 images
  Phytoxicity               → 223 images

chili:
  Antracnose                → 161 images
  Dieback                   → 105 images
  Healthy                   → 135 images

large_cardamom:
  Blight                    → 421 images
  Healthy                   → 385 images


In [ ]:
from google.colab import runtime
runtime.unassign()


In [ ]:
# ===============================================
# Install required packages (run once per session)
# ===============================================
!pip install -q timm seaborn pandas matplotlib scikit-learn tqdm torch torchvision

# ===============================================
# Mount Google Drive & set working directory
# ===============================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
from datetime import datetime

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Cross-Crop Transferability Index"
os.chdir(PROJECT_ROOT)
print("Current working directory:", os.getcwd())
print("Files in folder:", os.listdir('.'))

# Remove hidden checkpoints that break ImageFolder
!find "{PROJECT_ROOT}/data/test" -type d -name ".ipynb_checkpoints" -exec rm -rf {} + 2>/dev/null || true

# ===============================================
# Imports
# ===============================================
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader
from torchvision import datasets, models
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from collections import defaultdict
from tqdm import tqdm
import timm

# ================================
# CONFIG
# ================================
DATA_DIR = "data/test"
MODEL_DIR = "models"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

IMG_SIZE = 224
BATCH_SIZE = 96   # safe value for T4 + various model sizes

transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load crops safely
try:
    with open("crop_list.txt") as f:
        CROPS = [line.strip() for line in f if line.strip()]
    print("Crops to evaluate:", CROPS)
except Exception as e:
    print(f"Cannot read crop_list.txt → {e}")
    raise

# ================================
# CROP NAME DETECTION + FALLBACKS
# ================================
CROP_NAME_MAP = {
    "cauliflower": ["cauliflower", "cauli"],
    "chili": ["chilli", "chili"],
    "large_cardamom": ["cardamom", "suffle"],
    "maize": ["maize"],
    "mung_bean": ["mung", "mungbean"],
    "onion": ["onion"],
    "kidney_bean": ["rajma", "kidney"],
    "rice": ["rice"],
    "sesame": ["sesame"],
    "strawberry": ["strawberry", "strwaberry"],
    "Citrus": ["citrus", "Citrus"]
}

def get_crop_from_filename(filename):
    name = filename.lower()

    # First try exact keyword match
    for crop, keywords in CROP_NAME_MAP.items():
        if any(k in name for k in keywords):
            return crop

    # Fallback patterns for models that don't contain crop name
    if any(x in name for x in ["convnext", "v2plantnet", "atto"]):
        return "Citrus"           # ← change if this model was trained on different crop

    if "hybrid" in name and "maize" in name:
        return "maize"

    if "rice" in name:
        return "rice"

    if "strawberry" in name or "strwaberry" in name:
        return "strawberry"

    if "cauliflower" in name:
        return "cauliflower"

    return None

# ================================
# MODEL LOADING – FIXED DEVICE & HEAD ISSUE
# ================================
def load_pretrained_model(model_path, num_classes):
    filename = os.path.basename(model_path).lower()
    state_dict = torch.load(model_path, map_location=DEVICE, weights_only=True)

    print(f"Loading: {filename}  |  target classes: {num_classes}")

    # Create model and replace head BEFORE moving to device
    if any(k in filename for k in ["hybrid", "suffle", "maize", "mung", "onion", "chilli", "sesame", "squeez"]):
        try:
            from hybrid_model import HybridShuffleNetSqueezeNet
        except ImportError:
            raise ImportError("hybrid_model.py not found!")
        model = HybridShuffleNetSqueezeNet(num_classes=num_classes)

    elif "mobilenet" in filename or "mobile" in filename:
        model = models.mobilenet_v2(weights=None)
        in_f = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_f, num_classes)

    elif "resnet18" in filename:
        model = models.resnet18(weights=None)
        in_f = model.fc.in_features
        model.fc = nn.Linear(in_f, num_classes)

    elif "squeezenet" in filename or "squeez" in filename:
        model = models.squeezenet1_1(weights=None)
        model.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Conv2d(512, num_classes, 1),
            nn.ReLU(True),
            nn.AdaptiveAvgPool2d(1)
        )
        model.num_classes = num_classes

    elif any(x in filename for x in ["convnext", "v2", "atto", "v2plantnet"]):
        model_name = "convnextv2_atto"
        print(f"  → timm model: {model_name}")
        model = timm.create_model(model_name, pretrained=False, num_classes=num_classes)

    else:
        raise ValueError(f"Cannot detect model type from filename: {filename}")

    # Now move the whole model (with new head) to device
    model = model.to(DEVICE)

    # Load weights – ignore head mismatch
    model_dict = model.state_dict()
    new_sd = {}

    for k, v in state_dict.items():
        if k in model_dict and v.shape == model_dict[k].shape:
            new_sd[k] = v
            continue

        cleaned = k
        for p in ["module.", "model.", "net.", "student.", "teacher."]:
            if cleaned.startswith(p):
                cleaned = cleaned[len(p):]
                break

        if cleaned in model_dict and v.shape == model_dict[cleaned].shape:
            new_sd[cleaned] = v
            continue

        # Skip head keys explicitly
        if any(s in cleaned.lower() for s in ["head", "fc", "classifier"]):
            continue

    loaded = len(new_sd)
    total = len(model_dict)
    print(f"  → Loaded {loaded}/{total} keys ({loaded/total:.1%})")

    model.load_state_dict(new_sd, strict=False)
    model.eval()
    return model

# ================================
# EVALUATE
# ================================
def evaluate(model, dataloader):
    model.eval()
    preds, labels = [], []
    with torch.inference_mode():
        for imgs, lbls in tqdm(dataloader, desc="Infer", leave=False):
            imgs = imgs.to(DEVICE)
            outs = model(imgs)
            _, pred = torch.max(outs, 1)
            preds.extend(pred.cpu().numpy())
            labels.extend(lbls.cpu().numpy())
    return accuracy_score(labels, preds)

# ================================
# MAIN – COMPUTE CTI FOR ALL MODELS
# ================================
transfer_results = defaultdict(list)
transfer_matrix = pd.DataFrame(0.0, index=CROPS, columns=CROPS)

print("\nStarting CTI computation for ALL models in folder...\n")

for model_file in sorted(os.listdir(MODEL_DIR)):
    if not model_file.lower().endswith((".pth", ".pt")):
        continue

    model_path = os.path.join(MODEL_DIR, model_file)
    train_crop = get_crop_from_filename(model_file)

    if not train_crop:
        print(f"[SKIP] No crop detected → {model_file}")
        continue

    print(f"\nSource crop: {train_crop:18}   Model: {model_file}")

    for test_crop in CROPS:
        if test_crop == train_crop:
            continue

        test_path = os.path.join(DATA_DIR, test_crop)
        if not os.path.exists(test_path):
            print(f"  [MISSING] {test_crop}")
            continue

        try:
            dataset = datasets.ImageFolder(test_path, transform=transform)
            loader = DataLoader(
                dataset,
                batch_size=BATCH_SIZE,
                shuffle=False,
                num_workers=2,
                pin_memory=True,
                persistent_workers=True
            )

            num_classes = len(dataset.classes)
            model = load_pretrained_model(model_path, num_classes)
            acc = evaluate(model, loader)

            transfer_matrix.loc[train_crop, test_crop] = acc
            transfer_results[model_file].append((train_crop, test_crop, acc))
            print(f"  → {test_crop:18} : {acc:.4f}")

        except Exception as e:
            print(f"  [ERROR] {test_crop:18} → {str(e)}")

# ================================
# SAVE RESULTS WITH TIMESTAMP
# ================================
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

transfer_matrix.to_csv(f"cti_transfer_matrix_{ts}.csv")
print(f"\nSaved: cti_transfer_matrix_{ts}.csv")

cti_scores = {m: np.mean([a for _,_,a in trs]) if trs else 0.0
              for m, trs in transfer_results.items()}

cti_df = pd.DataFrame([
    {"Model": m, "CTI (%)": f"{v*100:.2f}"}
    for m, v in sorted(cti_scores.items(), key=lambda x: x[1], reverse=True)
])

cti_df.to_csv(f"cti_summary_{ts}.csv", index=False)

print("\n" + "═"*70)
print("CROSS-CROP TRANSFERABILITY INDEX (CTI)")
print("═"*70)
print(cti_df.to_string(index=False))
print("═"*70)

plt.figure(figsize=(12,10))
sns.heatmap(transfer_matrix, annot=True, fmt=".3f", cmap="viridis",
            cbar_kws={'label':'Accuracy'}, annot_kws={"size":9})
plt.title("Cross-Crop Transfer Accuracy (Source → Target)")
plt.ylabel("Train Crop")
plt.xlabel("Test Crop")
plt.tight_layout()
plt.savefig(f"cti_heatmap_{ts}.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: cti_heatmap_{ts}.png")

print("\nAll models processed. Check timestamped files.")

Mounted at /content/drive
Current working directory: /content/drive/MyDrive/Colab Notebooks/Cross-Crop Transferability Index
Files in folder: ['hybrid_model.py', 'crop_list.txt', 'compute_cti.py', 'cti_summary.csv', 'cti_transfer_matrix.csv', 'cti_heatmap.png', 'Untitled.ipynb', '__pycache__', 'models', '.ipynb_checkpoints', 'data', 'cti_transfer_matrix_20260222_221901.csv', 'cti_summary_20260222_221901.csv', 'cti_heatmap_20260222_221901.png']
Using device: cuda
Crops to evaluate: ['cauliflower', 'chili', 'large_cardamom', 'maize', 'mung_bean', 'onion', 'kidney_bean', 'rice', 'sesame', 'strawberry', 'Citrus']

Starting CTI computation for ALL models in folder...


Source crop: Citrus               Model: ConvNeXt_V2_Atto_best.pth
Loading: convnext_v2_atto_best.pth  |  target classes: 4
  → timm model: convnextv2_atto
  → Loaded 138/140 keys (98.6%)


  → cauliflower        : 0.2451
Loading: convnext_v2_atto_best.pth  |  target classes: 3
  → timm model: convnextv2_atto
  → Loaded 140/140 keys (100.0%)


  → chili              : 0.4165
Loading: convnext_v2_atto_best.pth  |  target classes: 2
  → timm model: convnextv2_atto
  → Loaded 138/140 keys (98.6%)


  → large_cardamom     : 0.4739
Loading: convnext_v2_atto_best.pth  |  target classes: 5
  → timm model: convnextv2_atto
  → Loaded 138/140 keys (98.6%)


  → maize              : 0.2895
Loading: convnext_v2_atto_best.pth  |  target classes: 2
  → timm model: convnextv2_atto
  → Loaded 138/140 keys (98.6%)


...

  → onion              : 0.4574
Loading: v2plantnet_state_dict.pth  |  target classes: 2
  → timm model: convnextv2_atto
  → Loaded 0/140 keys (0.0%)


  → kidney_bean        : 0.7444
Loading: v2plantnet_state_dict.pth  |  target classes: 5
  → timm model: convnextv2_atto
  → Loaded 0/140 keys (0.0%)


  → rice               : 0.2986
Loading: v2plantnet_state_dict.pth  |  target classes: 4
  → timm model: convnextv2_atto
  → Loaded 0/140 keys (0.0%)


  → sesame             : 0.3940
Loading: v2plantnet_state_dict.pth  |  target classes: 2
  → timm model: convnextv2_atto
  → Loaded 0/140 keys (0.0%)


  → strawberry         : 0.3430

Saved: cti_transfer_matrix_20260222_231502.csv

══════════════════════════════════════════════════════════════════════
CROSS-CROP TRANSFERABILITY INDEX (CTI)
══════════════════════════════════════════════════════════════════════
                                Model CTI (%)
                 HybridMaizemodel.pth   45.11
      ResNet18_Weights_Strawberry.pth   44.61
              squeezenet_Rajma_ND.pth   42.99
              Mobilenet_v2_Citrus.pth   41.05
    Squeez_suffle_cardamom_Hybrid.pth   39.65
          Sufflenet_v2_strwaberry.pth   39.61
            v2plantnet_state_dict.pth   39.29
            ConvNeXt_V2_Atto_best.pth   39.08
           MobilenetV2_Rice_model.pth   38.98
          Mobilenetv2_cauliflower.pth   37.64
                   Hybridmungbean.pth   37.15
                    hybrid_sesame.pth   35.71
Efficienet_net_V2model_Strawberry.pth   33.79
                    Hybrid_chilli.pth   33.09
               Hybrid_Onion_model.pth   30.70
══

In [ ]:
#To compute the Frech Inception Distance

In [ ]:
!ls "/content/drive/MyDrive/Colab Notebooks/Cross-Crop Transferability Index/data/test"

!ls "/content/drive/MyDrive/Colab Notebooks/Cross-Crop Transferability Index/models"
!ls "/content/drive/MyDrive/Colab Notebooks/Cross-Crop Transferability Index/data/test"

cauliflower  Citrus	  large_cardamom  mung_bean  rice    strawberry
chili	     kidney_bean  maize		  onion      sesame
ConvNeXt_V2_Atto_best.pth	       Mobilenet_v2_Citrus.pth
Efficienet_net_V2model_Strawberry.pth  MobilenetV2_Rice_model.pth
Hybrid_chilli.pth		       ResNet18_Weights_Strawberry.pth
HybridMaizemodel.pth		       squeezenet_Rajma_ND.pth
Hybridmungbean.pth		       Squeez_suffle_cardamom_Hybrid.pth
Hybrid_Onion_model.pth		       Sufflenet_v2_strwaberry.pth
hybrid_sesame.pth		       v2plantnet_state_dict.pth
Mobilenetv2_cauliflower.pth
cauliflower  Citrus	  large_cardamom  mung_bean  rice    strawberry
chili	     kidney_bean  maize		  onion      sesame


In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/Colab Notebooks/Cross-Crop Transferability Index"

print("BASE_PATH exists?         ", os.path.exists(BASE_PATH))
print("Is it a directory?        ", os.path.isdir(BASE_PATH))
print("test folder exists?       ", os.path.exists(f"{BASE_PATH} data/test"))
print("models folder exists?     ", os.path.exists(f"{BASE_PATH}/models"))
print("crop_list.txt exists?     ", os.path.exists(f"{BASE_PATH}/crop_list.txt"))
print("Number of items in folder:", len(os.listdir(BASE_PATH)))

# Bonus: list crop folders if test/ exists
if os.path.exists(f"{BASE_PATH}/test"):
    crops_found = [d for d in os.listdir(f"{BASE_PATH}/test") if os.path.isdir(f"{BASE_PATH}/test/{d}")]
    print("\nCrops found in test/:", sorted(crops_found))
    print("Number of crops:", len(crops_found))

BASE_PATH exists?          True
Is it a directory?         True
test folder exists?        False
models folder exists?      True
crop_list.txt exists?      True
Number of items in folder: 11


In [ ]:
# 2. Imports
# =============================================================================
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
import numpy as np
from scipy import linalg
import os
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# =============================================================================
# 3. Paths – CORRECTED FOR YOUR FOLDER STRUCTURE
# =============================================================================
BASE_PATH = "/content/drive/MyDrive/Colab Notebooks/Cross-Crop Transferability Index"
TEST_DIR    = f"{BASE_PATH}/data/test"          # ← your crop folders are here
FEATURES_DIR = f"{BASE_PATH}/features"          # will be created automatically
os.makedirs(FEATURES_DIR, exist_ok=True)

# Your 11 crops (exact names from your ls output – case sensitive!)
crops = [
    "cauliflower",
    "Citrus",
    "large_cardamom",
    "mung_bean",
    "rice",
    "strawberry",
    "chili",
    "kidney_bean",
    "maize",
    "onion",
    "sesame"
]

# =============================================================================
# 4. Load Inception-v3 for feature extraction
# =============================================================================
inception = models.inception_v3(pretrained=True, transform_input=False).eval()
inception.fc = nn.Identity()  # → 2048-dimensional features
inception = inception.to(device)

# Input transforms for Inception-v3
transform = transforms.Compose([
    transforms.Resize(299),
    transforms.CenterCrop(299),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# =============================================================================
# 5. Function: Extract and save features (runs only if not already saved)
# =============================================================================
def extract_and_save_features(crop_name, max_images=5000):
    feat_path = os.path.join(FEATURES_DIR, f"{crop_name}_features.npy")

    if os.path.exists(feat_path):
        print(f"→ Loaded existing features: {crop_name} ({feat_path})")
        return np.load(feat_path)

    folder = os.path.join(TEST_DIR, crop_name)
    if not os.path.exists(folder):
        print(f"× Folder not found: {folder}")
        return None

    dataset = ImageFolder(folder, transform=transform)
    print(f"→ {crop_name}: {len(dataset)} images found")

    # Subsample if too many images (helps with memory & stability)
    if len(dataset) > max_images:
        indices = np.random.choice(len(dataset), max_images, replace=False)
        dataset = Subset(dataset, indices)
        print(f"  → Subsampled to {max_images} images")

    loader = DataLoader(
        dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    features = []
    with torch.no_grad():
        for imgs, _ in tqdm(loader, desc=f"Extracting {crop_name}"):
            imgs = imgs.to(device)
            feats = inception(imgs).cpu().numpy()
            features.append(feats)

    if not features:
        print(f"× No features extracted for {crop_name}")
        return None

    features_array = np.concatenate(features, axis=0)
    np.save(feat_path, features_array)
    print(f"✓ Saved: {crop_name} → {features_array.shape} ({feat_path})")
    return features_array

# =============================================================================
# 6. Extract features for all crops
# =============================================================================
print("\n=== Extracting / Loading Features ===\n")
all_features = {}
for crop in crops:
    feats = extract_and_save_features(crop)
    if feats is not None:
        all_features[crop] = feats

print(f"\nTotal crops with features: {len(all_features)} / {len(crops)}\n")

# =============================================================================
# 7. FID computation function (with numerical stability)
# =============================================================================
def compute_fid(feats1, feats2, eps=1e-6):
    mu1 = np.mean(feats1, axis=0)
    mu2 = np.mean(feats2, axis=0)
    sigma1 = np.cov(feats1, rowvar=False)
    sigma2 = np.cov(feats2, rowvar=False)

    # Add small epsilon to diagonal to avoid singularity
    sigma1 += eps * np.eye(sigma1.shape[0])
    sigma2 += eps * np.eye(sigma2.shape[0])

    diff = mu1 - mu2
    covmean = linalg.sqrtm(sigma1.dot(sigma2)).real  # take real part if complex

    fid = diff.dot(diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * np.trace(covmean)
    return fid

# =============================================================================
# 8. Compute all cross-crop FIDs (excluding same-crop)
# =============================================================================
print("\n=== Computing Pairwise FID ===\n")
fids = []
pairs_count = 0

for src in crops:
    if src not in all_features:
        continue
    for tgt in crops:
        if tgt == src or tgt not in all_features:
            continue

        fid_value = compute_fid(all_features[src], all_features[tgt])
        print(f"FID  {src:15} → {tgt:15} : {fid_value:8.2f}")
        fids.append(fid_value)
        pairs_count += 1

if fids:
    avg_fid = np.mean(fids)
    min_fid = np.min(fids)
    max_fid = np.max(fids)
    print("\n" + "="*60)
    print(f" SUMMARY")
    print(f" Average FID (over {pairs_count} cross-crop pairs): {avg_fid:.2f}")
    print(f" Min FID: {min_fid:.2f}    Max FID: {max_fid:.2f}")
    print(f" (Lower FID = better feature distribution alignment)")
    print("="*60)
else:
    print("No valid cross-crop pairs could be computed.")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 199MB/s] 



=== Extracting / Loading Features ===

→ cauliflower: 869 images found


...

✓ Saved: maize → (2287, 2048) (/content/drive/MyDrive/Colab Notebooks/Cross-Crop Transferability Index/features/maize_features.npy)
→ onion: 258 images found


Extracting onion: 100%|██████████| 9/9 [00:37<00:00,  4.16s/it]


✓ Saved: onion → (258, 2048) (/content/drive/MyDrive/Colab Notebooks/Cross-Crop Transferability Index/features/onion_features.npy)
→ sesame: 434 images found


Extracting sesame: 100%|██████████| 14/14 [01:16<00:00,  5.45s/it]


✓ Saved: sesame → (434, 2048) (/content/drive/MyDrive/Colab Notebooks/Cross-Crop Transferability Index/features/sesame_features.npy)

Total crops with features: 11 / 11


=== Computing Pairwise FID ===

FID  cauliflower     → Citrus          :   261.91
FID  cauliflower     → large_cardamom  :   266.07
FID  cauliflower     → mung_bean       :   214.30
FID  cauliflower     → rice            :   321.35
FID  cauliflower     → strawberry      :   198.02
FID  cauliflower     → chili           :   251.98
FID  cauliflower     → kidney_bean     :   246.13
FID  cauliflower     → maize           :   250.19
FID  cauliflower     → onion           :   291.79
FID  cauliflower     → sesame          :   225.74
FID  Citrus          → cauliflower     :   261.91
FID  Citrus          → large_cardamom  :   209.62
FID  Citrus          → mung_bean       :   169.68
FID  Citrus          → rice            :   333.23
FID  Citrus          → strawberry      :   164.22
FID  Citrus          → chili           :   227.

In [ ]:
import pandas as pd
import numpy as np

# ────────────────────────────────────────────────────────────────
# Create pairwise DataFrame
# ────────────────────────────────────────────────────────────────

pairwise_rows = []

for src in crops:
    if src not in all_features:
        continue
    for tgt in crops:
        if tgt == src or tgt not in all_features:
            continue
        fid_value = compute_fid(all_features[src], all_features[tgt])
        pairwise_rows.append({
            'Source Crop': src,
            'Target Crop': tgt,
            'FID': round(fid_value, 2)
        })

df_pairwise = pd.DataFrame(pairwise_rows)

# ────────────────────────────────────────────────────────────────
# Summary statistics
# ────────────────────────────────────────────────────────────────

summary = {
    'Metric': [
        'Number of cross-crop pairs',
        'Average FID',
        'Minimum FID',
        'Maximum FID',
        'Median FID',
        'Standard Deviation'
    ],
    'Value': [
        len(df_pairwise),
        round(df_pairwise['FID'].mean(), 2),
        round(df_pairwise['FID'].min(), 2),
        round(df_pairwise['FID'].max(), 2),
        round(df_pairwise['FID'].median(), 2),
        round(df_pairwise['FID'].std(), 2)
    ]
}

df_summary = pd.DataFrame(summary)

# ────────────────────────────────────────────────────────────────
# Save to Excel (two sheets)
# ────────────────────────────────────────────────────────────────

excel_filename = f"FID_results_cross_crop_{pd.Timestamp.now().strftime('%Y%m%d_%H%M')}.xlsx"
excel_path = f"{BASE_PATH}/{excel_filename}"

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_summary.to_excel(writer, sheet_name='Summary', index=False)
    df_pairwise.to_excel(writer, sheet_name='Pairwise FID', index=False)

print(f"Excel file saved successfully:")
print(f"→ {excel_path}")
print(f"→ Sheets: 'Summary' and 'Pairwise FID'")

KeyboardInterrupt: 

In [ ]:
from google.colab import runtime
runtime.unassign()